# nlp-cimat at POLAR: Spanish Polarization Classification

This notebook implements an ensemble of Transformer models (RigoBERTa 2.0 and BETO) to classify attitude polarization in Spanish text. It includes custom text preprocessing, individual model fine-tuning, and a soft-voting ensemble for final inference.

### 1. Environment Setup and Reproducibility

We begin by installing the necessary Hugging Face libraries and setting up a global seed. Ensuring reproducibility is critical in NLP to maintain consistent results across different runs.

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
import random
import os, re
from transformers import set_seed
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Set the global seed value
SEED = 93330

# Fix seeds for basic Python and NumPy
random.seed(SEED)
np.random.seed(SEED)

# Fix seeds for PyTorch (CPU and GPU)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Set seed for Hugging Face Transformers
set_seed(SEED)

# Force deterministic behavior in CuDNN to ensure consistency
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### 2. Data Preprocessing

Social media text often contains noise (URLs, mentions, repeated characters). The `clean_text` function standardizes the input to help the models focus on the semantic content rather than formatting artifacts.

In [ ]:
def clean_text(t):
    """
    Standardizes and cleans raw text data using Regular Expressions.

    This pipeline removes noise common in social media and web data (URLs, mentions,
    non-printable characters) and normalizes linguistic variations (repeated letters,
    excessive punctuation) to reduce vocabulary sparsity.

    Args:
        t (str): The raw input string to be cleaned.

    Returns:
        str: The processed and normalized text string.
    """

    if not isinstance(t, str):
        return ""

    t = t.replace("\n", " ").replace("\t", " ").strip()
    t = re.sub(r'"+', '"', t)
    t = re.sub(r"http\S+|www\.\S+", " <url> ", t)
    t = re.sub(r"@\w+", " <user> ", t)
    t = re.sub(r"#(\w+)", r"\1", t)
    t = re.sub(r"(j+a+){2,}", "jaja", t, flags=re.IGNORECASE)
    t = re.sub(r"(.)\1{2,}", r"\1", t)
    t = "".join(ch for ch in t if ch.isprintable())
    t = re.sub(r"[·•●■□◆◇▢▣★☆▶▷◀◁♦▪]", " ", t)
    t = re.sub(r"[!?]{2,}", " ?", t)
    t = re.sub(r"[.]{2,}", ".", t)
    t = re.sub(r"\s+", " ", t).strip()

    return t

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/polar/train_augmented.csv")
dev_df   = pd.read_csv("/content/drive/MyDrive/polar/dev.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/polar/test.csv")

In [ ]:
train_df["text"] = train_df["text"].apply(clean_text)

In [ ]:
dev_df["text"] = dev_df["text"].apply(clean_text)

In [ ]:
test_df["text"]  = test_df["text"].apply(clean_text)

### 3. Training Pipeline

We define a modular training function. This allows us to fine-tune different Spanish LLMs using the same hyperparameters, ensuring a fair comparison and easier ensemble integration.

In [ ]:
def train_model(model_name, train_df, epoches):
    """
    Fine-tunes a Transformer-based model for binary sequence classification.

    Args:
        model_name (str): Pre-trained model identifier.
        train_df (pd.DataFrame): Dataframe with 'text' and 'polarization' columns.
        epoches (int): Number of training epochs.

    Returns:
        model, tokenizer: The fine-tuned model and its corresponding tokenizer.
    """
    print(f"\n--- Training model: {model_name} ---")

    # Load architecture-specific tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenization logic with fixed max length and truncation
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=160
        )

    # Prepare labels for the Trainer API
    train_df = train_df.rename(columns={"polarization": "labels"})

    # Convert to Hugging Face Dataset format
    dataset = Dataset.from_pandas(train_df)
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # Load model and move to active device (GPU/CPU)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # Define training hyperparameters
    args = TrainingArguments(
        output_dir=f"./{model_name.replace('/', '_')}",
        num_train_epochs=epoches,
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no",   # Optimization to save disk space
        report_to="none",     # Disable external logging
        seed=SEED             # Set seed for weight initialization
    )

    # Use data collator for dynamic sequence batching
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Initialize Trainer and start fine-tuning
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=data_collator
    )

    trainer.train()

    return model, tokenizer

In [ ]:
MODEL_RIGOBERTA = "IIC/RigoBERTa-2.0"
MODEL_BETO =  "dccuchile/bert-base-spanish-wwm-cased"

In [ ]:
model_rigoberta, tok_rigoberta = train_model(MODEL_RIGOBERTA, train_df, 3)

In [ ]:
model_beto, tok_beto = train_model(MODEL_BETO, train_df, 3)

### 4. Ensemble Prediction and Development Set Evaluation

To improve robustness, we implement a soft-voting ensemble that averages the prediction probabilities of all models. This section evaluates the performance using the Development Set, providing F1-score and Accuracy.

In [ ]:
def predict_ensemble(test_df, models, tokenizers):
    """
    Performs soft-voting ensemble inference by averaging class probabilities
    across multiple fine-tuned models.

    Args:
        test_df (pd.DataFrame): Dataframe containing the 'text' column for inference.
        models (list): List of trained AutoModelForSequenceClassification instances.
        tokenizers (list): List of corresponding tokenizers for each model.

    Returns:
        np.array: Final class predictions based on the highest averaged probability.
    """
    probs_total = None

    for model, tokenizer in zip(models, tokenizers):
        # Tokenize the entire text column and move tensors to the active device
        enc = tokenizer(
            list(test_df["text"]),
            truncation=True,
            padding=True,
            max_length=160,
            return_tensors="pt"
        ).to(device)

        # Disable gradient calculation for faster inference and lower memory usage
        with torch.no_grad():
            logits = model(**enc).logits
            # Convert raw logits to probabilities using Softmax
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        # Accumulate probabilities for averaging
        if probs_total is None:
            probs_total = probs
        else:
            probs_total += probs

    # Calculate the mean probability across all models (Soft-Voting)
    probs_total /= len(models)

    # Select the class with the highest average probability
    preds = np.argmax(probs_total, axis=1)
    return preds

In [ ]:
# Set both models to evaluation mode
model_rigoberta.eval()
model_beto.eval()

# Generate predictions using the ensemble on development data
preds = predict_ensemble(
    dev_df,
    models=[model_rigoberta, model_beto],
    tokenizers=[tok_rigoberta, tok_beto]
)

# Calculate performance metrics
accuracy = accuracy_score(dev_df["polarization"], preds)
f1 = f1_score(dev_df["polarization"], preds)

# Print final results
print("=== Results ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}\n")

### 5. Final Inference on the Unlabeled Test Set

Finally, we apply the ensemble to the official Test Set (unlabeled data). We process the hidden samples and generate the submission file containing only the ID and the predicted polarization.

In [ ]:
def predict_ensemble_unlabeled(
    df,
    models,
    tokenizers,
    text_col="text",
    return_proba=False
):
    """
    Generates predictions for unlabeled test data using the model ensemble.

    Args:
        df (pd.DataFrame): The input dataframe containing raw text.
        models (list): Trained models to be used for inference.
        tokenizers (list): Tokenizers corresponding to each model.
        text_col (str): Name of the column containing the text data.
        return_proba (bool): Whether to return raw probabilities (default: False).

    Returns:
        np.array: Predicted class indices for the test set.
    """
    probs_total = None

    for model, tokenizer in zip(models, tokenizers):
        # Set model to evaluation mode to disable dropout/batchnorm layers
        model.eval()

        # Tokenize input text for the current model architecture
        enc = tokenizer(
            list(df[text_col]),
            truncation=True,
            padding=True,
            max_length=160,
            return_tensors="pt"
        ).to(device)

        # Execute forward pass without gradient tracking
        with torch.no_grad():
            logits = model(**enc).logits
            # Convert logits to normalized probability distributions
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        # Aggregate probabilities across all ensemble members
        probs_total = probs if probs_total is None else probs_total + probs

    # Compute the average probability across the ensemble
    probs_total /= len(models)

    # Map the averaged probabilities to the final class prediction
    preds = np.argmax(probs_total, axis=1)

    return preds


In [ ]:
# Set models to evaluation mode (disables dropout)
model_beto.eval()
model_rigo.eval()

# Generate ensemble predictions on the unlabeled test data
preds = predict_ensemble_unlabeled(
    unlabeled_df,
    models=[model_beto, model_rigo],
    tokenizers=[tok_beto, tok_rigo]
)

# Prepare the output dataframe with IDs and predicted labels
output_df = unlabeled_df[["id"]].copy()
output_df["polarization"] = preds

# Save the final results to a CSV file for submission
output_df.to_csv(
    "/content/drive/MyDrive/polar/spa.csv",
    index=False
)

print("Process complete. Submission file saved.")